In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os, csv, random
from collections import defaultdict
 
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, f1_score
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm
 


In [8]:
!git clone --depth 1 https://huggingface.co/datasets/ViratGarg/animal_species_SMAI

Cloning into 'animal_species_SMAI'...
remote: Enumerating objects: 26517, done.
remote: Counting objects: 100% (26517/26517), done.
remote: Compressing objects: 100% (26517/26517), done.
remote: Total 26517 (delta 0), reused 26487 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26517/26517), 3.66 MiB | 16.57 MiB/s, done.
Updating files: 100% (26661/26661), done.
Filtering content: 100% (26659/26659), 1.86 GiB | 2.88 MiB/s, done.


In [3]:
DATA_DIR   = "/kaggle/working/animal_species_SMAI"
OUTPUT_DIR = "/kaggle/working/hier_outputs"
BATCH_SIZE = 32
S1_EPOCHS  = 10    # group classifier  (3 classes)
S2_EPOCHS  = 15    # species classifiers
LR         = 1e-4
IMG_SIZE   = 224
VAL_RATIO  = 0.10
TEST_RATIO = 0.10
SEED       = 42
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
 
print(f"device: {DEVICE}\n")
 


device: cuda



In [4]:

# ── class / group definitions ─────────────────────────────────────────────────
ORDERED_CLASS_NAMES = [
    # butterflies / moths  (indices 0–29)
    "CAIRNS BIRDWING", "BLUE SPOTTED CROW", "AMERICAN SNOOT", "BECKERS WHITE",
    "BLACK HAIRSTREAK", "BANDED PEACOCK", "BROWN SIPROETA", "BIRD CHERRY ERMINE MOTH",
    "CABBAGE WHITE", "BANDED TIGER MOTH", "APPOLLO", "CLEARWING MOTH",
    "ATALA", "CLEOPATRA", "AFRICAN GIANT SWALLOWTAIL", "CLOUDED SULPHUR",
    "ATLAS MOTH", "ADONIS", "BANDED ORANGE HELICONIAN", "ARCIGERA FLOWER MOTH",
    "BROOKES BIRDWING", "CHECQUERED SKIPPER", "CLODIUS PARNASSIAN", "CINNABAR MOTH",
    "COMET MOTH", "CHALK HILL BLUE", "BLUE MORPHO", "CHESTNUT", "AN 88", "BROWN ARGUS",
    # mammals  (indices 30–74)
    "warthog", "vampire_bat", "highland_cattle", "seal", "badger",
    "baboon", "horse", "rhinoceros", "arctic_fox", "wildebeest",
    "opossum", "orangutan", "polar_bear", "blue_whale", "jackal",
    "vicuna", "manatee", "otter", "mountain_goat", "yak",
    "squirrel", "giraffe", "porcupine", "weasel", "dolphin",
    "brown_bear", "zebra", "camel", "tapir", "alpaca",
    "snow_leopard", "sugar_glider", "kangaroo", "sea_lion", "red_panda",
    "african_elephant", "walrus", "american_bison", "koala", "mongoose",
    "wombat", "groundhog", "armadillo", "anteater", "water_buffalo",
    # birds  (indices 75–99)
    "Forest Wagtail", "Northern Lapwing", "Rufous Treepie", "Cattle Egret",
    "Common Kingfisher", "Gray Wagtail", "Indian Peacock", "House Crow",
    "White-Breasted Waterhen", "Indian Pitta", "Red-Wattled Lapwing",
    "White-Breasted Kingfisher", "Indian Roller", "Common Tailorbird",
    "White Wagtail", "Common Rosefinch", "Jungle Babbler", "Coppersmith Barbet",
    "Hoopoe", "Sarus Crane", "Common Myna", "Brown-Headed Barbet",
    "Ruddy Shelduck", "Indian Grey Hornbill", "Asian Green Bee-Eater",
]
 
NAME_TO_IDX = {n.lower(): i for i, n in enumerate(ORDERED_CLASS_NAMES)}
 
GROUP_MAP = {
    **{i: "butterfly" for i in range(0,  30)},
    **{i: "mammals"   for i in range(30, 75)},
    **{i: "birds"     for i in range(75, 100)},
}
GROUP_TO_IDX = {"birds": 0, "butterfly": 1, "mammals": 2}
IDX_TO_GROUP = {v: k for k, v in GROUP_TO_IDX.items()}
 
# build per-group species lists and local↔global index maps
group_species = defaultdict(list)
for sp_idx in range(len(ORDERED_CLASS_NAMES)):
    group_species[GROUP_MAP[sp_idx]].append(sp_idx)
 
sp_to_local     = {}            # global_sp_idx  → local idx within group
local_to_global = {}            # (group, local)  → global_sp_idx
for group, sp_list in group_species.items():
    for local_idx, global_idx in enumerate(sp_list):
        sp_to_local[global_idx]               = local_idx
        local_to_global[(group, local_idx)]   = global_idx
 
 


In [5]:

# ── scan dataset from disk ────────────────────────────────────────────────────
all_paths   = []
all_species = []   # global species index
skipped     = []
 
for root, _, files in os.walk(DATA_DIR):
    if root == DATA_DIR:
        continue
    folder = os.path.basename(root)
    imgs   = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if not imgs:
        continue
    idx = NAME_TO_IDX.get(folder.lower())
    if idx is None:
        skipped.append(folder)
        continue
    for img in imgs:
        all_paths.append(os.path.join(root, img))
        all_species.append(idx)
 
all_species = np.array(all_species)
all_groups  = np.array([GROUP_TO_IDX[GROUP_MAP[s]] for s in all_species])
 
print(f"total images : {len(all_paths)}")
if skipped:
    print(f"skipped folders: {skipped}")
for g, sp_list in group_species.items():
    n = int((all_groups == GROUP_TO_IDX[g]).sum())
    print(f"  {g:12s}: {len(sp_list):3d} species | {n:5d} images")
 
 


total images : 26659
  butterfly   :  30 species |  4158 images
  mammals     :  45 species | 13751 images
  birds       :  25 species |  8750 images


In [6]:

# ── stratified train / val / test split (on species) ─────────────────────────
all_idx = list(range(len(all_paths)))
train_idx, temp_idx = train_test_split(
    all_idx, test_size=(VAL_RATIO + TEST_RATIO),
    stratify=all_species, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=all_species[temp_idx],
    random_state=SEED
)
print(f"\nsplit → train:{len(train_idx)}  val:{len(val_idx)}  test:{len(test_idx)}\n")
 



split → train:21327  val:2666  test:2666



In [7]:

# ── transforms ────────────────────────────────────────────────────────────────
_mean, _std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
 
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(_mean, _std),
])
eval_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(_mean, _std),
])
 
 


In [8]:

# ── dataset wrappers ──────────────────────────────────────────────────────────
class Stage1Dataset(Dataset):
    """image → group label  (0=birds, 1=butterfly, 2=mammals)"""
    def __init__(self, indices, transform=None):
        self.indices   = indices
        self.transform = transform
 
    def __len__(self):
        return len(self.indices)
 
    def __getitem__(self, i):
        ri  = self.indices[i]
        img = Image.open(all_paths[ri]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(all_groups[ri])
 
 
class Stage2Dataset(Dataset):
    """image → local species label, filtered to one group"""
    def __init__(self, indices, group_name, transform=None):
        self.transform = transform
        # keep only images belonging to this group
        self.indices   = [i for i in indices
                          if GROUP_MAP[all_species[i]] == group_name]
        self.group     = group_name
 
    def __len__(self):
        return len(self.indices)
 
    def __getitem__(self, i):
        ri  = self.indices[i]
        img = Image.open(all_paths[ri]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, sp_to_local[all_species[ri]]
 
 
class PipelineDataset(Dataset):
    """returns (image, global_species_label, group_label) for end-to-end eval"""
    def __init__(self, indices, transform=None):
        self.indices   = indices
        self.transform = transform
 
    def __len__(self):
        return len(self.indices)
 
    def __getitem__(self, i):
        ri  = self.indices[i]
        img = Image.open(all_paths[ri]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(all_species[ri]), int(all_groups[ri])
 
 


In [9]:

# ── model ─────────────────────────────────────────────────────────────────────
def build_resnet(num_classes):
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    m.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(m.fc.in_features, num_classes)
    )
    return m
 
 
# ── training helpers ──────────────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer, device, phase):
    model.train() if phase == "train" else model.eval()
    total_loss = 0.0
    all_true, all_pred = [], []
 
    with torch.set_grad_enabled(phase == "train"):
        for batch in tqdm(loader, desc=f"  {phase:5s}", leave=False):
            imgs, labels = batch[0].to(device), batch[1].to(device)
            if phase == "train":
                optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if phase == "train":
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            all_true.extend(labels.cpu().numpy())
            all_pred.extend(logits.argmax(1).cpu().numpy())
 
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(all_true), np.array(all_pred)
 
 
def train_model(model, loaders, criterion, optimizer, scheduler,
                n_epochs, device, save_path):
    best_val_acc = 0.0
    for ep in range(1, n_epochs + 1):
        lr_now = scheduler.get_last_lr()[0]
        print(f"  epoch {ep:02d}/{n_epochs}  lr={lr_now:.2e}")
 
        tr_loss, tr_true, tr_pred = run_epoch(
            model, loaders["train"], criterion, optimizer, device, "train")
        vl_loss, vl_true, vl_pred = run_epoch(
            model, loaders["val"], criterion, None, device, "val")
        scheduler.step()
 
        tr_acc = accuracy_score(tr_true, tr_pred)
        vl_acc = accuracy_score(vl_true, vl_pred)
        print(f"    train  loss={tr_loss:.4f}  acc={tr_acc:.4f}")
        print(f"    val    loss={vl_loss:.4f}  acc={vl_acc:.4f}")
 
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), save_path)
            print(f"    ✓ checkpoint saved  (val_acc={vl_acc:.4f})")
 
    print(f"  best val acc: {best_val_acc:.4f}\n")
    return best_val_acc
 
 
def batch_predict(model, loader, device):
    """run inference, return (y_true, y_pred) arrays"""
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="  eval", leave=False):
            imgs, labels = batch[0].to(device), batch[1].to(device)
            all_true.extend(labels.cpu().numpy())
            all_pred.extend(model(imgs).argmax(1).cpu().numpy())
    return np.array(all_true), np.array(all_pred)
 
 


In [10]:

# ── metrics ───────────────────────────────────────────────────────────────────
def macro_specificity(y_true, y_pred, num_classes):
    cm    = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    specs = []
    total = cm.sum()
    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i].sum()  - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return float(np.mean(specs))
 
 
def print_metrics(y_true, y_pred, label_names, title=""):
    n   = len(label_names)
    acc = accuracy_score(y_true, y_pred)
    mf1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    msp = macro_specificity(y_true, y_pred, n)
 
    print(f"\n{'─'*62}")
    if title:
        print(f"  {title}")
    print(f"  accuracy          : {acc:.4f}")
    print(f"  macro F1          : {mf1:.4f}")
    print(f"  macro specificity : {msp:.4f}")
    print()
    print(classification_report(y_true, y_pred,
                                target_names=label_names,
                                zero_division=0, digits=3))
    return {"accuracy": acc, "macro_f1": mf1, "macro_specificity": msp}
 


In [11]:

# ════════════════════════════════════════════════════════════════════════════
#  STAGE 1 — group classifier  (3 classes: birds | butterfly | mammals)
# ════════════════════════════════════════════════════════════════════════════
print("═"*62)
print("  STAGE 1 — group classifier  (3 classes)")
print("═"*62 + "\n")
 
s1_loaders = {
    "train": DataLoader(Stage1Dataset(train_idx, train_tfm),
                        batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True),
    "val":   DataLoader(Stage1Dataset(val_idx, eval_tfm),
         
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True),
}
s1_test_loader = DataLoader(Stage1Dataset(test_idx, eval_tfm),
                            batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)
 
s1_model  = build_resnet(3).to(DEVICE)
s1_crit   = nn.CrossEntropyLoss(label_smoothing=0.1)
s1_optim  = optim.AdamW(s1_model.parameters(), lr=LR, weight_decay=1e-4)
s1_sched  = optim.lr_scheduler.CosineAnnealingLR(s1_optim, T_max=S1_EPOCHS)
s1_path   = os.path.join(OUTPUT_DIR, "stage1_best.pth")
 
train_model(s1_model, s1_loaders, s1_crit, s1_optim, s1_sched,
            S1_EPOCHS, DEVICE, s1_path)
 
s1_model.load_state_dict(torch.load(s1_path, map_location=DEVICE))
s1_true, s1_pred = batch_predict(s1_model, s1_test_loader, DEVICE)
s1_metrics = print_metrics(s1_true, s1_pred,
                            label_names=["birds", "butterfly", "mammals"],
                            title="Stage 1 — test results")
 


══════════════════════════════════════════════════════════════
  STAGE 1 — group classifier  (3 classes)
══════════════════════════════════════════════════════════════

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 190MB/s] 


  epoch 01/10  lr=1.00e-04


    train  loss=0.3280  acc=0.9891
    val    loss=0.2996  acc=0.9974
    ✓ checkpoint saved  (val_acc=0.9974)
  epoch 02/10  lr=9.76e-05


    train  loss=0.2995  acc=0.9983
    val    loss=0.2960  acc=0.9985
    ✓ checkpoint saved  (val_acc=0.9985)
  epoch 03/10  lr=9.05e-05


...

    train  loss=0.2931  acc=0.9999
    val    loss=0.2936  acc=0.9992
  epoch 08/10  lr=2.06e-05


    train  loss=0.2928  acc=0.9999
    val    loss=0.2931  acc=0.9992
  epoch 09/10  lr=9.55e-06


    train  loss=0.2928  acc=0.9998
    val    loss=0.2925  acc=0.9992
  epoch 10/10  lr=2.45e-06


    train  loss=0.2926  acc=1.0000
    val    loss=0.2928  acc=0.9989
  best val acc: 0.9996




──────────────────────────────────────────────────────────────
  Stage 1 — test results
  accuracy          : 0.9989
  macro F1          : 0.9986
  macro specificity : 0.9993

              precision    recall  f1-score   support

       birds      1.000     0.999     0.999       875
   butterfly      0.998     0.998     0.998       415
     mammals      0.999     0.999     0.999      1376

    accuracy                          0.999      2666
   macro avg      0.999     0.999     0.999      2666
weighted avg      0.999     0.999     0.999      2666



In [12]:
# ════════════════════════════════════════════════════════════════════════════
#  STAGE 2 — one species classifier per group
# ════════════════════════════════════════════════════════════════════════════
print("═"*62)
print("  STAGE 2 — species classifiers")
print("═"*62)
 
s2_models      = {}
s2_metrics_all = {}
 
for group_name in ["birds", "butterfly", "mammals"]:
    n_species   = len(group_species[group_name])
    local_names = [ORDERED_CLASS_NAMES[g] for g in group_species[group_name]]
 
    print(f"\n── {group_name.upper()}  ({n_species} species) ──\n")
 
    s2_loaders = {
        "train": DataLoader(Stage2Dataset(train_idx, group_name, train_tfm),
                            batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True),
        "val":   DataLoader(Stage2Dataset(val_idx, group_name, eval_tfm),
                            batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True),
    }
    s2_test_loader = DataLoader(
        Stage2Dataset(test_idx, group_name, eval_tfm),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True
    )
 
    model  = build_resnet(n_species).to(DEVICE)
    crit   = nn.CrossEntropyLoss(label_smoothing=0.1)
    optim_ = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched_ = optim.lr_scheduler.CosineAnnealingLR(optim_, T_max=S2_EPOCHS)
    path   = os.path.join(OUTPUT_DIR, f"stage2_{group_name}_best.pth")
 
    train_model(model, s2_loaders, crit, optim_, sched_,
                S2_EPOCHS, DEVICE, path)
 
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    s2_true, s2_pred = batch_predict(model, s2_test_loader, DEVICE)
    metrics = print_metrics(s2_true, s2_pred,
                            label_names=local_names,
                            title=f"Stage 2 — {group_name} test results")
 
    s2_models[group_name]      = model
    s2_metrics_all[group_name] = metrics
 
 


══════════════════════════════════════════════════════════════
  STAGE 2 — species classifiers
══════════════════════════════════════════════════════════════

── BIRDS  (25 species) ──

  epoch 01/15  lr=1.00e-04


    train  loss=1.5258  acc=0.7554
    val    loss=0.8250  acc=0.9349
    ✓ checkpoint saved  (val_acc=0.9349)
  epoch 02/15  lr=9.89e-05


    train  loss=0.7573  acc=0.9637
    val    loss=0.7771  acc=0.9474
    ✓ checkpoint saved  (val_acc=0.9474)
  epoch 03/15  lr=9.57e-05


    train  loss=0.7148  acc=0.9759
    val    loss=0.7407  acc=0.9646
    ✓ checkpoint saved  (val_acc=0.9646)
  epoch 04/15  lr=9.05e-05


    train  loss=0.6820  acc=0.9874
    val    loss=0.7257  acc=0.9669
    ✓ checkpoint saved  (val_acc=0.9669)
  epoch 05/15  lr=8.35e-05


...

    train  loss=0.7188  acc=0.9972
    val    loss=0.7960  acc=0.9680
  epoch 13/15  lr=9.55e-06


    train  loss=0.7178  acc=0.9971
    val    loss=0.7918  acc=0.9738
    ✓ checkpoint saved  (val_acc=0.9738)
  epoch 14/15  lr=4.32e-06


    train  loss=0.7155  acc=0.9977
    val    loss=0.7911  acc=0.9724
  epoch 15/15  lr=1.09e-06


    train  loss=0.7160  acc=0.9973
    val    loss=0.7912  acc=0.9702
  best val acc: 0.9738




──────────────────────────────────────────────────────────────
  Stage 2 — mammals test results
  accuracy          : 0.9666
  macro F1          : 0.9659
  macro specificity : 0.9992

                  precision    recall  f1-score   support

         warthog      1.000     1.000     1.000        26
     vampire_bat      1.000     0.958     0.979        24
 highland_cattle      0.967     0.935     0.951        31
            seal      0.923     0.750     0.828        32
          badger      0.933     0.903     0.918        31
          baboon      0.943     1.000     0.971        33
           horse      1.000     0.967     0.983        30
      rhinoceros      1.000     1.000     1.000        28
      arctic_fox      1.000     0.935     0.967        31
      wildebeest      0.969     1.000     0.984        31
         opossum      0.971     1.000     0.985        33
       orangutan      1.000     0.971     0.985        34
      polar_bear      0.946     0.972     0.959        36
  

In [13]:

# ════════════════════════════════════════════════════════════════════════════
#  END-TO-END PIPELINE EVAL
#  Stage 1 predicts the group → that group's Stage 2 model predicts species
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*62)
print("  END-TO-END PIPELINE EVALUATION")
print("═"*62 + "\n")
 
pipeline_loader = DataLoader(
    PipelineDataset(test_idx, eval_tfm),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
 
s1_model.eval()
for m in s2_models.values():
    m.eval()
 
pipe_true    = []   # global species index
pipe_pred    = []   # global species index (via s1→s2 routing)
group_hits   = 0
 
with torch.no_grad():
    for imgs, sp_true, grp_true in tqdm(pipeline_loader, desc="  pipeline"):
        imgs = imgs.to(DEVICE)
 
        grp_logits = s1_model(imgs)
        grp_pred   = grp_logits.argmax(1)
 
        # process each sample individually (routing depends on predicted group)
        for b in range(imgs.size(0)):
            img_b   = imgs[b:b+1]
            g_pred  = IDX_TO_GROUP[grp_pred[b].item()]
            g_true  = IDX_TO_GROUP[grp_true[b].item()]
 
            if g_pred == g_true:
                group_hits += 1
 
            local_pred  = s2_models[g_pred](img_b).argmax(1).item()
            global_pred = local_to_global[(g_pred, local_pred)]
 
            pipe_true.append(sp_true[b].item())
            pipe_pred.append(global_pred)
 
pipe_true = np.array(pipe_true)
pipe_pred = np.array(pipe_pred)
 
group_acc    = group_hits / len(test_idx)
pipeline_acc = accuracy_score(pipe_true, pipe_pred)
pipeline_f1  = f1_score(pipe_true, pipe_pred, average="macro", zero_division=0)
pipeline_sp  = macro_specificity(pipe_true, pipe_pred, len(ORDERED_CLASS_NAMES))
 
print(f"  group routing accuracy   : {group_acc:.4f}")
print(f"  end-to-end species acc   : {pipeline_acc:.4f}")
print(f"  end-to-end macro F1      : {pipeline_f1:.4f}")
print(f"  end-to-end macro spec    : {pipeline_sp:.4f}")
 
# per-group breakdown of pipeline performance
print(f"\n  per-group pipeline breakdown:")
test_groups_arr = all_groups[test_idx]   # same order as pipe_true / pipe_pred
for group_name in ["birds", "butterfly", "mammals"]:
    g_idx = GROUP_TO_IDX[group_name]
    mask  = test_groups_arr == g_idx
    if mask.sum() == 0:
        continue
    g_acc = accuracy_score(pipe_true[mask], pipe_pred[mask])
    g_f1  = f1_score(pipe_true[mask], pipe_pred[mask],
                     average="macro", zero_division=0)
    g_sp  = macro_specificity(pipe_true[mask], pipe_pred[mask],
                              len(ORDERED_CLASS_NAMES))
    print(f"    {group_name:12s}  acc={g_acc:.4f}  macro_f1={g_f1:.4f}"
          f"  macro_spec={g_sp:.4f}  (n={mask.sum()})")
 
 



══════════════════════════════════════════════════════════════
  END-TO-END PIPELINE EVALUATION
══════════════════════════════════════════════════════════════



  pipeline: 100%|██████████| 84/84 [00:26<00:00,  3.20it/s]

  group routing accuracy   : 0.9989
  end-to-end species acc   : 0.9689
  end-to-end macro F1      : 0.9691
  end-to-end macro spec    : 0.9997

  per-group pipeline breakdown:
    birds         acc=0.9714  macro_f1=0.9345  macro_spec=0.9997  (n=875)
    butterfly     acc=0.9735  macro_f1=0.9428  macro_spec=0.9997  (n=415)
    mammals       acc=0.9658  macro_f1=0.9446  macro_spec=0.9997  (n=1376)


In [14]:

# ── error analysis: misrouted samples ────────────────────────────────────────
misrouted_mask = test_groups_arr != all_groups[np.array(
    [test_idx[i] for i in range(len(test_idx))]
)]
# simpler: compare stage1 ground truth vs pipeline routing
s1_true_all = all_groups[test_idx]
# we already know group_hits, so:
n_misrouted = len(test_idx) - group_hits
print(f"\n  samples misrouted by stage 1 : {n_misrouted}"
      f"  ({100*n_misrouted/len(test_idx):.1f}%)")
print(f"  of those, correct species still possible only if routed correctly\n")
 



  samples misrouted by stage 1 : 3  (0.1%)
  of those, correct species still possible only if routed correctly



In [15]:

# ── save summary ──────────────────────────────────────────────────────────────
summary = {
    "s1_test_acc":        round(s1_metrics["accuracy"], 4),
    "s1_test_macro_f1":   round(s1_metrics["macro_f1"], 4),
    "s1_test_macro_spec": round(s1_metrics["macro_specificity"], 4),
    "pipeline_group_acc": round(group_acc, 4),
    "pipeline_sp_acc":    round(pipeline_acc, 4),
    "pipeline_macro_f1":  round(pipeline_f1, 4),
    "pipeline_macro_spec":round(pipeline_sp, 4),
}
for g, met in s2_metrics_all.items():
    summary[f"s2_{g}_acc"]        = round(met["accuracy"], 4)
    summary[f"s2_{g}_macro_f1"]   = round(met["macro_f1"], 4)
    summary[f"s2_{g}_macro_spec"] = round(met["macro_specificity"], 4)
 
csv_path = os.path.join(OUTPUT_DIR, "hierarchical_metrics.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=summary.keys())
    w.writeheader()
    w.writerow(summary)
 
print(f"  metrics saved → {csv_path}")
print(f"\n  checkpoints:")
print(f"    {OUTPUT_DIR}/stage1_best.pth")
for g in ["birds", "butterfly", "mammals"]:
    print(f"    {OUTPUT_DIR}/stage2_{g}_best.pth")


  metrics saved → /kaggle/working/hier_outputs/hierarchical_metrics.csv

  checkpoints:
    /kaggle/working/hier_outputs/stage1_best.pth
    /kaggle/working/hier_outputs/stage2_birds_best.pth
    /kaggle/working/hier_outputs/stage2_butterfly_best.pth
    /kaggle/working/hier_outputs/stage2_mammals_best.pth
